# Machine Learning based on Manual Reference (ML-MR) Phenotyping

This notebook implements the manual-reference ML workflow used by the SPACEMAP GUI. The publication-facing configuration uses two disjoint manual-reference columns:

- `phenotype_ref1a`: training labels and the sole source used to estimate per-method F1 weights.
- `phenotype_ref1b`: held-out labels used only after fitting to evaluate the final weighted MR-ML ensemble.

The method combines:

1. **Supervised classification** trained on marker expression from the training-reference cells.
2. **Weighted ensemble integration** using per-class F1 scores calculated only from the training reference.
3. **Held-out-cell final-output validation** against the row-disjoint reference, with accuracy, balanced accuracy, macro F1, weighted F1, and a confusion matrix.
4. **Ensemble-support scoring** for every prediction (useful for uncertainty ranking, but not a calibrated probability).

The workflow prepares `phenotype_ml_mr`, `ml_mr_confidence`, and the refined output for review in memory. The GUI writes selected assignments to the unified table only after **Save All Assignments to Disk** is clicked. Reference columns must never be selected as ensemble methods, because that would leak the answer into the prediction.

This Ref1a/Ref1b design prevents reuse of the exact same cells for fitting and validation. Because the two subsets can still contain cells from the same patients and ROIs, it is a held-out-cell validation—not a patient-level external validation. A separate patient, section, or curated Ref3 set remains the stronger publication validation.


## Step 1: Configuration and Setup

Configure the ML-MR analysis parameters including file paths, methods to integrate, and model settings.

In [ ]:
from pathlib import Path
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Setup paths and imports
project_root = Path().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from spacemap.ml_mr_utils import run_ml_mr_analysis

# =============================================================================
# CONFIGURATION
# =============================================================================

# Active V16 project and enriched unified table (contains Ref1a/Ref1b)
ACTIVE_PROJECT_DIR = project_root / 'project' / 'INNATE_4_IMAGES_07302026'
UNIFIED_CSV = ACTIVE_PROJECT_DIR / 'outputs' / 'batch_corrected' / 'batch_corrected_merged_with_ref_enriched.csv'

# Output directory
OUTPUT_DIR = ACTIVE_PROJECT_DIR / 'outputs' / 'phenotyping' / 'ML_MR'

# Analysis parameters
GROUND_TRUTH_COLUMN = 'phenotype_ref1a'  # Training labels and F1-weight estimation only
VALIDATION_GROUND_TRUTH_COLUMN = 'phenotype_ref1b'  # Held-out final-ensemble validation; set to None for exploratory internal split
OUTPUT_PHENOTYPE_COL = 'phenotype_ml_mr'
CONFIDENCE_COL_NAME = 'ml_mr_confidence'
REFINED_OUTPUT_PHENOTYPE_COL = 'phenotype_ml_mr_refined'
REFINED_CONFIDENCE_THRESHOLD = 0.75
REFINED_LOW_CONFIDENCE_LABEL = 'Others'
ML_METHOD = 'random_forest'
ANALYSIS_FILTER_COLUMN = 'tissue'  # Restrict fitting/validation/prediction to eligible tissue cells
ANALYSIS_FILTER_VALUE = 1
DROP_INVALID_MARKER_ROWS = True  # Match the GUI complete-marker eligibility check
WRITE_BACKUP_CSV = False  # Review-first default: do not create an automatic result CSV
UPDATE_SOURCE_CSV = False  # Save explicitly only after reviewing assignments

# Marker configuration
CORR_MARKERS = [
    "CD31", "E-cadherin", "CD20", "CD4", "Pan-Cytokeratin", "CD68", "Podoplanin", "CD11c", "CD8", "HLA-DR",
    "CD3e", "CD45", "CD15", "FoxP3", "PDGFRa", "aSMA", "CD163", "FN1", "S100A9", "CD11b", "CD38",
    "Vimentin"
]

NON_CORR_MARKERS = [
    # Add non-corrected markers here if needed
]

# Combined marker list for clustering
MARKERS_BASE = CORR_MARKERS + NON_CORR_MARKERS
CLUSTERING_MARKERS_CORR = [f"{m}_corr" for m in CORR_MARKERS] + NON_CORR_MARKERS

# New: Morphology features (transformed: log1p for sizes/lengths, sin/cos for orientation; then Z-norm if enabled)
# Purpose: Incorporate cell shape/context for better phenotype discrimination.
# Usage: List exact column names from unified CSV; set to [] to disable.
MORPHOLOGY_COLS = []

# New: Compartment binaries (0/1 indicators; optionally down-weighted for soft context)
# Purpose: Provide spatial priors without hard-gating, preserving rare infiltrates.
# Usage: List exact binary column names; set to ['tumor', 'epithelia', 'stroma', 'endothelia'] to disable.
COMPARTMENT_COLS = []  # Adjust to your column names

# Feature-block controls. A value of 0 excludes a block. Positive rescaling does not
# alter tree splits and is removed by the SVM's internal standardization, so use
# these values mainly to include (nonzero) or exclude (zero) feature blocks.
FEATURE_WEIGHTS = {'markers': 1.0, 'morphology': 0, 'compartments': 0}

# Z-normalization parameters
# Corrected markers have already passed upstream batch correction; leave this False
# for the primary analysis unless a different, pre-specified protocol requires it.
USE_Z_NORMALIZATION = False
BATCH_NORMALIZATION_COL = 'batch_id'  # Column to group by for Z-normalization

# Phenotyping methods to integrate
PHENOTYPING_METHODS = [
    'phenotype_leiden',
    'phenotype_som',
    'phenotype_threshold',
    'phenotype_vortex'
]
EXCLUDED_PHENOTYPES = ['Others']  # Ignored as reference targets and automated-method votes

# Model parameters
TEST_SIZE = 0.3  # Used only when VALIDATION_GROUND_TRUTH_COLUMN is None
N_ESTIMATORS = 1500  # Number of trees in Random Forest
RANDOM_STATE = 42  # For reproducibility

print("ML-MR Analysis Configuration:")
print(f"- Input unified CSV: {UNIFIED_CSV}")
print(f"- Output directory: {OUTPUT_DIR}")
print(f"- Training reference column: {GROUND_TRUTH_COLUMN}")
print(f"- Held-out validation column: {VALIDATION_GROUND_TRUTH_COLUMN or 'internal split'}")
print(f"- Refined output: {REFINED_OUTPUT_PHENOTYPE_COL} (support < {REFINED_CONFIDENCE_THRESHOLD:g} -> {REFINED_LOW_CONFIDENCE_LABEL})")
print(f"- Analysis population: {ANALYSIS_FILTER_COLUMN} == {ANALYSIS_FILTER_VALUE!r}")
print(f"- Drop rows with invalid selected markers: {DROP_INVALID_MARKER_ROWS}")
print(f"- Automatic result CSV: {WRITE_BACKUP_CSV}")
print(f"- Update source CSV during run: {UPDATE_SOURCE_CSV}")
print(f"- Selected markers: {len(CLUSTERING_MARKERS_CORR)} total")
print(f"  * Corrected markers: {len(CORR_MARKERS)}")
print(f"  * Non-corrected markers: {len(NON_CORR_MARKERS)}")
print(f"- Morphology features: {MORPHOLOGY_COLS} (transformed and Z-norm if enabled)")
print(f"- Compartment binaries: {COMPARTMENT_COLS} (weighted as soft context)")
print(f"- Feature weights: {FEATURE_WEIGHTS}")
print(f"- Z-normalization: {USE_Z_NORMALIZATION} (grouped by '{BATCH_NORMALIZATION_COL}')")
print(f"- Methods to integrate: {PHENOTYPING_METHODS}")
print(f"- Phenotypes excluded from ML-MR: {EXCLUDED_PHENOTYPES}")
validation_text = (f'held-out column {VALIDATION_GROUND_TRUTH_COLUMN}' if VALIDATION_GROUND_TRUTH_COLUMN else f'{TEST_SIZE:.0%} internal split')
print(f"- Model parameters: {ML_METHOD}, {N_ESTIMATORS} estimators, validation={validation_text}")


## Step 2: Data Validation and Exploration

Examine the input data to understand the training set characteristics and method performance.

In [ ]:
# Load and validate the unified dataset
print("Loading and examining input data...")
source_df = pd.read_csv(UNIFIED_CSV)
print(f"Unified dataset loaded: {len(source_df):,} cells, {len(source_df.columns)} columns")
if ANALYSIS_FILTER_COLUMN not in source_df.columns:
    raise KeyError(f"Analysis filter column not found: {ANALYSIS_FILTER_COLUMN}")
analysis_mask = source_df[ANALYSIS_FILTER_COLUMN].astype('string').str.strip().eq(str(ANALYSIS_FILTER_VALUE))
unified_df = source_df.loc[analysis_mask].copy()
if unified_df.empty:
    raise ValueError('The configured analysis filter selected no rows')
print(f"Analysis population: {len(unified_df):,}/{len(source_df):,} rows")
existing_markers = [column for column in CLUSTERING_MARKERS_CORR if column in unified_df.columns]
missing_markers = [column for column in CLUSTERING_MARKERS_CORR if column not in unified_df.columns]
print(f"Selected markers found: {len(existing_markers)}/{len(CLUSTERING_MARKERS_CORR)}")
if missing_markers:
    raise KeyError(f"Missing selected markers: {missing_markers}")
if DROP_INVALID_MARKER_ROWS:
    marker_values = unified_df[existing_markers].apply(pd.to_numeric, errors='coerce')
    finite_marker_mask = np.isfinite(marker_values.to_numpy()).all(axis=1)
    dropped = int((~finite_marker_mask).sum())
    unified_df = unified_df.loc[finite_marker_mask].copy()
    print(f"Complete-marker analysis population: {len(unified_df):,} rows ({dropped:,} excluded)")

INVALID_LABELS = {"", "unclassified", "unknown", "nan", "none"} | {str(label).strip().lower() for label in EXCLUDED_PHENOTYPES}
def usable_reference_mask(series):
    values = series.astype("string").str.strip()
    return series.notna() & ~values.str.lower().isin(INVALID_LABELS)

required_reference_columns = [GROUND_TRUTH_COLUMN]
if VALIDATION_GROUND_TRUTH_COLUMN:
    required_reference_columns.append(VALIDATION_GROUND_TRUTH_COLUMN)
missing_references = [column for column in required_reference_columns if column not in unified_df.columns]
if missing_references:
    raise KeyError(f"Missing reference columns: {missing_references}")

training_mask = usable_reference_mask(unified_df[GROUND_TRUTH_COLUMN])
print(f"Training reference ({GROUND_TRUTH_COLUMN}): {int(training_mask.sum()):,} usable labels")
print(unified_df.loc[training_mask, GROUND_TRUTH_COLUMN].value_counts().sort_index())

if VALIDATION_GROUND_TRUTH_COLUMN:
    validation_mask = usable_reference_mask(unified_df[VALIDATION_GROUND_TRUTH_COLUMN])
    overlap = int((training_mask & validation_mask).sum())
    print(f"Held-out validation ({VALIDATION_GROUND_TRUTH_COLUMN}): {int(validation_mask.sum()):,} usable labels")
    print(unified_df.loc[validation_mask, VALIDATION_GROUND_TRUTH_COLUMN].value_counts().sort_index())
    if overlap:
        raise ValueError(f"Leakage detected: {overlap:,} rows are labeled in both reference columns")
    print("Reference-row overlap: 0 (required)")

reference_like_methods = [
    method for method in PHENOTYPING_METHODS
    if method.startswith("phenotype_ref") or method in {"phenotype_bd", "phenotype_ap"}
]
if reference_like_methods:
    raise ValueError(f"Manual references cannot be ensemble methods: {reference_like_methods}")
missing_methods = [method for method in PHENOTYPING_METHODS if method not in unified_df.columns]
if missing_methods:
    raise KeyError(f"Missing phenotyping methods: {missing_methods}")
print(f"Automated methods to integrate: {PHENOTYPING_METHODS}")
print("Method F1 weights will be calculated from the training reference only.")

if USE_Z_NORMALIZATION:
    if BATCH_NORMALIZATION_COL not in unified_df.columns:
        raise KeyError(f"Batch normalization column not found: {BATCH_NORMALIZATION_COL}")
    print(f"Additional batch-wise Z-normalization enabled: {BATCH_NORMALIZATION_COL}")
else:
    print("Additional Z-normalization disabled; using upstream batch-corrected marker values.")


## Step 3: Execute ML-MR Analysis

Run the complete ML-MR analysis workflow including model training, prediction, and evaluation.

In [ ]:
# Run the ML-MR analysis
print("=" * 80)
print("EXECUTING ML-MR ANALYSIS")
print("=" * 80)

RESULT_HOLDER = {}
def capture_results(result_frame):
    RESULT_HOLDER['dataframe'] = result_frame

success = run_ml_mr_analysis(
    unified_csv_path=UNIFIED_CSV,
    performance_csv_path=None,
    output_dir=OUTPUT_DIR,
    ground_truth_column=GROUND_TRUTH_COLUMN,
    validation_ground_truth_column=VALIDATION_GROUND_TRUTH_COLUMN,
    analysis_filter_column=ANALYSIS_FILTER_COLUMN,
    analysis_filter_value=ANALYSIS_FILTER_VALUE,
    drop_invalid_marker_rows=DROP_INVALID_MARKER_ROWS,
    phenotyping_methods=PHENOTYPING_METHODS,
    clustering_markers=CLUSTERING_MARKERS_CORR,
    morphology_cols=MORPHOLOGY_COLS,
    compartment_cols=COMPARTMENT_COLS,
    feature_weights=FEATURE_WEIGHTS,
    use_z_normalization=USE_Z_NORMALIZATION,
    batch_normalization_col=BATCH_NORMALIZATION_COL,
    test_size=TEST_SIZE,
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE,
    ml_method=ML_METHOD,
    output_phenotype_col=OUTPUT_PHENOTYPE_COL,
    confidence_col_name=CONFIDENCE_COL_NAME,
    refined_output_phenotype_col=REFINED_OUTPUT_PHENOTYPE_COL,
    refined_confidence_threshold=REFINED_CONFIDENCE_THRESHOLD,
    refined_low_confidence_label=REFINED_LOW_CONFIDENCE_LABEL,
    excluded_phenotypes=EXCLUDED_PHENOTYPES,
    write_backup_csv=WRITE_BACKUP_CSV,
    update_source_csv=UPDATE_SOURCE_CSV,
    results_callback=capture_results,
)

if success:
    print("\nML-MR analysis completed successfully!")
    print(f"Check the output directory for detailed results: {OUTPUT_DIR}")
else:
    print("\nML-MR analysis failed. Check error messages above.")


## Step 4: Results Summary and Validation

Examine the ML-MR results and compare with existing phenotyping methods.

In [ ]:
# Examine in-memory results before any assignment save
print("Examining ML-MR results...")
updated_df = RESULT_HOLDER.get('dataframe')
if updated_df is None:
    raise RuntimeError('No in-memory ML-MR results were returned')
required_outputs = [OUTPUT_PHENOTYPE_COL, CONFIDENCE_COL_NAME, REFINED_OUTPUT_PHENOTYPE_COL]
missing_outputs = [column for column in required_outputs if column not in updated_df.columns]
if missing_outputs:
    raise RuntimeError(f"Missing ML-MR output columns: {missing_outputs}")

result_mask = (
    updated_df[ANALYSIS_FILTER_COLUMN].astype('string').str.strip().eq(str(ANALYSIS_FILTER_VALUE))
    & updated_df[OUTPUT_PHENOTYPE_COL].notna()
)
analysis_results = updated_df.loc[result_mask].copy()
print(f"Predictions: {analysis_results[OUTPUT_PHENOTYPE_COL].notna().sum():,} analysis cells")
print()
print("ML-MR phenotype distribution:")
print(analysis_results[OUTPUT_PHENOTYPE_COL].value_counts().head(15))
print()
print("Ensemble-support score summary (not a calibrated probability):")
print(analysis_results[CONFIDENCE_COL_NAME].describe())
low_confidence_mask = analysis_results[CONFIDENCE_COL_NAME] < REFINED_CONFIDENCE_THRESHOLD
print(f"Low-confidence cells relabeled in refined output: {int(low_confidence_mask.sum()):,}")
print(f"Refined phenotype distribution ({REFINED_OUTPUT_PHENOTYPE_COL}):")
print(analysis_results[REFINED_OUTPUT_PHENOTYPE_COL].value_counts().head(15))

reference_for_reporting = VALIDATION_GROUND_TRUTH_COLUMN or GROUND_TRUTH_COLUMN
report_mask = usable_reference_mask(analysis_results[reference_for_reporting])
report_df = analysis_results.loc[report_mask]
print()
print(f"Validation reference: {reference_for_reporting} ({len(report_df):,} cells)")
methods_to_compare = [OUTPUT_PHENOTYPE_COL] + [
    method for method in PHENOTYPING_METHODS if method in updated_df.columns
]
for method in methods_to_compare:
    valid = report_df[method].notna()
    if valid.any():
        accuracy = (report_df.loc[valid, reference_for_reporting] == report_df.loc[valid, method]).mean()
        print(f"  {method}: {accuracy:.3f} accuracy ({int(valid.sum()):,} cells)")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].hist(analysis_results[CONFIDENCE_COL_NAME].dropna(), bins=50, alpha=0.7, edgecolor='black')
axes[0].set(xlabel='ML-MR ensemble-support score', ylabel='Cells', title='Support-score distribution')
counts = analysis_results[OUTPUT_PHENOTYPE_COL].value_counts().head(10).sort_values()
axes[1].barh(counts.index, counts.values)
axes[1].set(xlabel='Cells', title='Top ML-MR phenotypes')
plt.tight_layout()
plot_path = OUTPUT_DIR / 'ml_mr_summary_plot.png'
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Summary plot saved to: {plot_path}")
plt.show()


## Step 5: Output Summary

Summary of generated files and next steps for using the ML-MR results.

In [ ]:
# Display summary of outputs and next steps
print("=" * 80)
print("ML-MR ANALYSIS SUMMARY")
print("=" * 80)

print(f"\nGenerated Files and Outputs:")
print(f"Output directory: {OUTPUT_DIR}")

# List generated files
if OUTPUT_DIR.exists():
    output_files = list(OUTPUT_DIR.rglob('*'))
    if output_files:
        print(f"\nGenerated files:")
        for file_path in sorted(output_files):
            if file_path.is_file():
                file_size = file_path.stat().st_size / 1024  # KB
                print(f"   - {file_path.name} ({file_size:.1f} KB)")
    else:
        print(f"\nNo output files found in {OUTPUT_DIR}")

print(f"\nReview-first assignment state:")
print(f"   - Original unified CSV unchanged: {not UPDATE_SOURCE_CSV}")
print(f"   - Source path: {UNIFIED_CSV}")
print(f"   - In-memory columns ready for review:")
print(f"     * {OUTPUT_PHENOTYPE_COL}: ML-MR phenotype predictions")
print(f"     * {CONFIDENCE_COL_NAME}: Ensemble-support scores (0-1; not calibrated probabilities)")
print(f"     * {REFINED_OUTPUT_PHENOTYPE_COL}: predictions with support below {REFINED_CONFIDENCE_THRESHOLD:g} labeled '{REFINED_LOW_CONFIDENCE_LABEL}'")

print(f"\nKey Features of Enhanced ML-MR Method:")
print(f"   - Trained and weighted only on {GROUND_TRUTH_COLUMN}")
print(f"   - Final ensemble validated on {VALIDATION_GROUND_TRUTH_COLUMN or 'an internal exploratory split'}")
print(f"   - Comprehensive feature set:")
print(f"     * Markers: {len(CLUSTERING_MARKERS_CORR)} curated markers")
print(f"       - Corrected markers: {len(CORR_MARKERS)}")
print(f"       - Non-corrected markers: {len(NON_CORR_MARKERS)}")
print(f"     * Morphology: {len(MORPHOLOGY_COLS)} features (transformed)")
print(f"       - Size/length features: log1p transformed")
print(f"       - Orientation: sin/cos transformed")
print(f"     * Compartments: {len(COMPARTMENT_COLS)} spatial binaries")
print(f"   - Feature weighting: {FEATURE_WEIGHTS}")
print(f"   - Z-normalization: {'Enabled' if USE_Z_NORMALIZATION else 'Disabled'}")
if USE_Z_NORMALIZATION:
    print(f"     * Grouped by: '{BATCH_NORMALIZATION_COL}' column")
print("   - Per-class F1 weighting: computed from manual reference")
print(f"   - Integrates {len(PHENOTYPING_METHODS)} existing phenotyping methods")
print(f"   - Uses F1-score based weighting for method integration")
print(f"   - Provides ensemble-support scores for uncertainty ranking")
print(f"   - Based on Random Forest with {N_ESTIMATORS} trees")

print(f"\nNext Steps:")
print(f"   1. Review the validation report: {OUTPUT_DIR}/ml_mr_validation_report.txt")
print(f"   2. Examine performance heatmaps in the output directory")
print(f"   3. Analyze confidence-stratified results for quality assessment")
print(f"   4. Use '{OUTPUT_PHENOTYPE_COL}' for downstream analyses")
print(f"   5. Use '{REFINED_OUTPUT_PHENOTYPE_COL}' when a conservative low-support category is required")
print(f"   6. Review '{CONFIDENCE_COL_NAME}' for uncertainty summaries")
print(f"   7. Compare with consensus ML results from Step9B")

print(f"\nEnhanced ML-MR analysis complete!")
print("=" * 80)
